# Summary

**Para ver la key**:

Para ver o gestionar tu clave de API (API Key) de Google Gemini, debes acceder al centro de control de desarrolladores de Google AI. Sigue estos pasos:

1.  **Entra a Google AI Studio:** Dirígete a [aistudio.google.com](https://aistudio.google.com/).
2.  **Inicia sesión:** Usa la misma cuenta de Google con la que creaste el proyecto original.
3.  **Sección de API Keys:** En el menú lateral izquierdo (normalmente un icono de llave o un botón que dice **"Get API key"**), haz clic.
4.  **Lista de Keys:** Verás una tabla con las claves que has creado. 
    * Si ya tienes una, aparecerá como `My API Key`. Puedes hacer clic en **"Copy"** para copiarla al portapapeles.
    * Si no ves ninguna, haz clic en el botón azul **"Create API key in new project"**.

---

Luego en la raiz del proyecto crean el archivo:

`.env`

Ahi lo pegan de esta manera:

GOOGLE_API_KEY='``KEY``' <br>
GEMINI_API_KEY='``KEY``'


In [114]:
# imports
# Load environment variables in a file called .env
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

# Librerias de uso
import json


In [115]:
from openai import OpenAI
import os
import requests

# Elegir si gemini (con quota) o ollama
elegir_llm = 'ollama'


if elegir_llm == 'gemini':
    modelo = "gemini-2.5-flash"
    gemini = OpenAI(
        api_key=os.getenv("GOOGLE_API_KEY"),
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
    )
    response = gemini.chat.completions.create(model=modelo, 
                                    messages=[{"role":"user", "content": "what is 2+2?"}])
    print(response.choices[0].message.content)

elif elegir_llm == 'ollama':
    # Verificar que Ollama está corriendo
    try:
        requests.get("http://localhost:11434", timeout=2)
        print("✅ Ollama funcionando")
    except:
        print("❌ Primero ejecuta 'ollama serve' en terminal")
        exit()

    import subprocess
    # Descargar el modelo
    result = subprocess.run(["ollama", "pull", "llama3.2"], capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)

    # Configurar cliente
    modelo = "llama3.2"
    OLLAMA_BASE_URL = "http://localhost:11434/v1"
    gemini = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

    # Función reutilizable
    def preguntar_ollama(prompt, modelo=modelo):
        response = gemini.chat.completions.create(
            model=modelo, 
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content

    # Usarlo
    respuesta = preguntar_ollama("Tell me a fun fact")
    print(respuesta)

✅ Ollama funcionando


Exception in thread Thread-128 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\Pc\anaconda3\Lib\threading.py", line 1041, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "C:\Users\Pc\anaconda3\Lib\threading.py", line 992, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Pc\anaconda3\Lib\subprocess.py", line 1609, in _readerthread
    buffer.append(fh.read())
                  ~~~~~~~^^
  File "C:\Users\Pc\anaconda3\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8f in position 405: character maps to <undefined>



None
Here's one: 

Did you know that there is a species of jellyfish that is immortal? The Turritopsis dohrnii, also known as the "immortal jellyfish," can transform its body into a younger state through a process called transdifferentiation. This means it can essentially revert back to its polyp stage and grow back into an adult again, allowing it to bypass the normal process of aging and death. Isn't that just mind-blowing?


# Modelos y sus outputs

En el contexto de **LLMs** (Modelos de Lenguaje Grande) los campos `role` y `content` tienen estos significados:

**`role`** (rol)<br>
Indica **quién** está generando o a quién pertenece el mensaje. Los roles más comunes son:

- **`"system"`(prompt)** - Instrucciones del sistema (define el comportamiento, personalidad o reglas del asistente)
- **`"user"`** - Mensaje del usuario final (la pregunta o instrucción)
- **`"assistant"`** - Respuesta generada por el modelo (útil para mantener historial de conversación)
- **`"function"`** (o `"tool"`) - Resultado de llamadas a funciones/herramientas

**`content`** (contenido)<br>
Es el **texto real** del mensaje. Puede ser:
- Un string simple: `"¿Cómo estás?"`
- Una lista (para mensajes multimodales): `[{"type": "text", "text": "..."}, {"type": "image_url", "image_url": {...}}]`

**Ejemplo práctico en Python**

```python
mensajes = [
    {"role": "system", "content": "Eres un asistente útil que responde en español."},
    {"role": "user", "content": "¿Cuál es la capital de Francia?"},
    {"role": "assistant", "content": "La capital de Francia es París."},
    {"role": "user", "content": "¿Y su población?"}]
```

In [116]:
mensaje = 'Cuanto es 2 + 2?'

# Construyo el diccionario
messages_chat = [{"role": "user", "content": mensaje}]

# Se lo paso al modelo
response = gemini.chat.completions.create(
        model=modelo, 
        messages=messages_chat
    )

# Imprimimos la respuesta. Usamos json.dumps() para que se vea bonita la respuesta:
print(json.dumps(response.model_dump(), indent=2))

{
  "id": "chatcmpl-576",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "La respuesta es: 4.",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": null
      }
    }
  ],
  "created": 1775926210,
  "model": "llama3.2",
  "object": "chat.completion",
  "service_tier": null,
  "system_fingerprint": "fp_ollama",
  "usage": {
    "completion_tokens": 8,
    "prompt_tokens": 34,
    "total_tokens": 42,
    "completion_tokens_details": null,
    "prompt_tokens_details": null
  }
}


Es bastante info, por lo que vamos a quedarnos con lo importante:

In [117]:
# Después de obtener response
print(f"content: {response.choices[0].message.content}")
print(f"role: {response.choices[0].message.role}")
print(f"function_call: {response.choices[0].message.function_call}")
print(f"tool_calls: {response.choices[0].message.tool_calls}")
print(f"completion_tokens: {response.usage.completion_tokens}")
print(f"prompt_tokens: {response.usage.prompt_tokens}")
print(f"total_tokens: {response.usage.total_tokens}")

content: La respuesta es: 4.
role: assistant
function_call: None
tool_calls: None
completion_tokens: 8
prompt_tokens: 34
total_tokens: 42


| Campo | Valor en tu respuesta | ¿Qué significa? |
|-------|----------------------|-----------------|
| **content** | `"2 + 2 es **4**."` | Es el texto real que el modelo generó como respuesta. Aquí está la respuesta a tu pregunta. Puede incluir formato como markdown, código, emojis, etc. Es lo que normalmente le mostrarías al usuario. |
| **role** | `"assistant"` | Indica quién produjo este mensaje. `"assistant"` significa que fue generado por el modelo de IA. Otros roles posibles son `"user"` (el humano que pregunta) o `"system"` (instrucciones del sistema). Esto ayuda a mantener el orden en conversaciones multi-turno. |
| **function_call** | `None` | Cuando el modelo necesita obtener información externa (clima, base de datos, cálculos complejos), en lugar de responder directamente, devuelve una solicitud para llamar a una función. Contiene el nombre de la función y sus argumentos. `None` significa que el modelo respondió directamente sin necesitar funciones externas. |
| **tool_calls** | `None` | Es la versión moderna y más potente de `function_call`. Permite al modelo usar múltiples herramientas simultáneamente (ej: buscar en web + calcular + leer archivo). Es más flexible y escalable. `None` indica que no se solicitó ninguna herramienta. |
| **completion_tokens** | `8` | Número de tokens (fragmentos de texto) que el modelo generó en su respuesta. Los tokens son como "palabras" o "partes de palabras" que el modelo procesa. Por ejemplo, "2", "+", "2", "es", "**", "4", "**", "." = 8 tokens. Esto determina el costo de la respuesta. |
| **prompt_tokens** | `10` | Número de tokens que tú enviaste como mensaje de entrada. En este caso, "what is 2+2?" se convierte en 10 tokens para el modelo. Esto también se cobra en las APIs. |
| **total_tokens** | `86` | Suma total de todos los tokens procesados: `prompt_tokens` (10) + `completion_tokens` (8) = 18, pero aquí ves 86. **¿Por qué?** Porque los modelos modernos también cuentan tokens internos de procesamiento, metadatos, y en este caso específico de Gemini, incluye el `thought_signature` (un token especial de razonamiento interno). Este valor es el que realmente se usa para facturación. |

**Nota importante sobre `total_tokens`:**
En Gemini (y algunos otros modelos avanzados), hay tokens adicionales internos que se suman al total, como:
- Tokens de razonamiento interno
- Firma de pensamiento (`thought_signature`)
- Metadatos de procesamiento

**Por eso:** 10 + 8 = 18, pero tu `total_tokens` es 86. Los 68 tokens extra son internos del modelo.

## Historial de conversacion - Ilusion de Memoria

**Historial de conversación**	
Es la lista de mensajes previos que le envías al modelo en cada petición. 
Ejemplo: 
```python
[{"role": "user", "content": "Hola"}, {"role": "assistant", "content": "¿Cómo estás?"}, {"role": "user", "content": "Bien"}]. 
```
El modelo NO guarda nada entre llamadas, así que tú debes enviarle todo el historial cada vez.

**Ilusión de Memoria**
El modelo NO tiene memoria real. Cada vez que hablas con él, es como si fuera la primera vez. Pero al enviarle el historial completo, el modelo simula recordar la conversación anterior. Eso es una "ilusión": parece que recuerda, pero en realidad solo está procesando el texto que le reenvías.

In [118]:
mensaje = 'Me llamo Juan'
messages_chat = [{"role": "user", "content": mensaje}]

# Se lo paso al modelo
response = gemini.chat.completions.create(model=modelo, messages=messages_chat)
print(response.choices[0].message.content)

¡Hola Juan! Me alegra conocerte. ¿Cómo estás hoy? ¿En qué puedo ayudarte o querés charlar un rato? Estoy aquí para escucharte y ayudarte en lo que necesites.


In [119]:
mensaje = "¿Cómo me llamo?"
messages_chat = [{"role": "user", "content": mensaje}]

# Se lo paso al modelo
response = gemini.chat.completions.create(model=modelo, messages=messages_chat)
print(response.choices[0].message.content)


No te preocupes, pero no tengo un nombre específico. Soy una inteligencia artificial diseñada para ayudar a las personas con sus preguntas y problemas. Me llaman "asistente" o simplemente "AI", pero puedes llamarme lo que prefieras.

¿En qué puedo ayudarte hoy?


Como se ve, no tiene memoria. Vamos ahora a:
- Pasarle prompt
- Pasarle el nombre
- Guardar su respuesta
- Preguntar por nuestro nombre
- Ver su respuesta 

In [120]:
# Primero definimos la primera parte
propmt = 'Eres un agente que aprende nombres'
mensaje_1 = 'Mi nombre es Akali'

messages_chat = [
    {"role": "system", "content": propmt},
    {"role": "user", "content": mensaje_1},
]
response_1 = gemini.chat.completions.create(
    model=modelo, 
    messages=messages_chat
)
respuesta_1 = response_1.choices[0].message.content


# ------------------------------------------------------------------------#

# Ahora que tenemos la historia, agregamos la respuesta y luego la pregunta
mensaje_2 = "¿Cómo me llamo?"

messages_chat = [
    {"role": "system", "content": propmt},
    {"role": "user", "content": mensaje_1},
    {"role": "assistant", "content": respuesta_1},
    {"role": "user", "content": mensaje_2}
]

# Pasamos de nuevo TODO el historial actualizado al modelo
response_2 = gemini.chat.completions.create(
    model=modelo, 
    messages=messages_chat
)

# 4. Ver su respuesta
respuesta_2 = response_2.choices[0].message.content
print(respuesta_2)


Disculpa la confusión, Akali. Si no recuerdo mal, tu nombre es "Akali". Sin embargo, el miembro que tengo en la base de conocimientos con un nombre similar es... Xialing y también Daka  no sé si los dos nombres que tiene son iguales al tuyo.

 Pero te puedo decir una verdad, Akali nombre en chino está siendo buscado por los inteligencia.


## Simular un chat con historial

Tenemos un historial con mucha información que el modelo **no necesita**.

Vamos a limpiarla con un loop

In [121]:
# Este es el historial que RECIBE la función (con basura)
history = [
    {"role": "user",        "content": "Hola",                  "timestamp": 1700000000,  "user_id": 123,  "tokens": 111},
    {"role": "assistant",   "content": "Hola",                  "timestamp": 1888887200,  "user_id": 456,  "tokens": 223},
    {"role": "user",        "content": "Te amo UwU",            "timestamp": 1999999920,  "user_id": 789,  "tokens": 555},
    {"role": "assistant",   "content": "¿En que te ayudo?",     "timestamp": 1888887200,  "user_id": 456,  "tokens": 223},
]


# Tenemos que limpiarlo para pasarlo al LLM
history_limpio = []
for hist in history:  
    diccionario_limpio_temporal = {
        "role": hist["role"],           # Primero tenemos que pasar el rol de cada linea
        "content": hist["content"]}     # luego quedarnos con el contenido 

    print(diccionario_limpio_temporal, "\n")  # Vemos que se guardo bien

    history_limpio.append(diccionario_limpio_temporal)  


print('\nImprimimos el resultado')
print(history_limpio)

{'role': 'user', 'content': 'Hola'} 

{'role': 'assistant', 'content': 'Hola'} 

{'role': 'user', 'content': 'Te amo UwU'} 

{'role': 'assistant', 'content': '¿En que te ayudo?'} 


Imprimimos el resultado
[{'role': 'user', 'content': 'Hola'}, {'role': 'assistant', 'content': 'Hola'}, {'role': 'user', 'content': 'Te amo UwU'}, {'role': 'assistant', 'content': '¿En que te ayudo?'}]


Ahora vamos a escribirlo mas simple para ponerlo en una funcion

In [122]:
# Este es el historial que RECIBE la función (con basura)
history = [
    {"role": "user",        "content": "Hola",                  "timestamp": 1700000000,  "user_id": 123,  "tokens": 111},
    {"role": "assistant",   "content": "Hola",                  "timestamp": 1888887200,  "user_id": 456,  "tokens": 223},
    {"role": "user",        "content": "Te amo UwU",            "timestamp": 1999999920,  "user_id": 789,  "tokens": 555},
    {"role": "assistant",   "content": "¿En que te ayudo?",     "timestamp": 1888887200,  "user_id": 456,  "tokens": 223},
]

# Tenemos que limpiarlo para pasarlo al LLM
history_limpio_2 = [{"role":h["role"], "content":h["content"]} for h in history]

print('\nImprimimos el resultado')
print(history_limpio_2)


Imprimimos el resultado
[{'role': 'user', 'content': 'Hola'}, {'role': 'assistant', 'content': 'Hola'}, {'role': 'user', 'content': 'Te amo UwU'}, {'role': 'assistant', 'content': '¿En que te ayudo?'}]


### Funcion ``Chat_historial``
Ahora creamos una funcion que tiene historial y hace todo por nosotros

In [123]:
def chat_historial( message, history):

    # Hacemos la lista limpia del historial
    history = [{"role":h["role"], "content":h["content"]} for h in history]

    # Construimos el mensaje con prompt, historial y mensaje que le escribimos
    messages_chat = [{"role": "system", "content": prompt_inicial}] + history + [{"role": "user", "content": message}]

    # Corremos el modelo
    response = gemini.chat.completions.create(model=modelo, messages=messages_chat)

    # Guardamos la respuesta
    respuesta = response.choices[0].message.content

    # Limpiamos la respuesta
    return print(respuesta.encode().decode())

In [124]:
prompt_inicial = 'Sos OTP Akali. Siempre recomiendas jugar ese champ'  # Esta variable no es necesario pasarla a la funcion porque accede directamente (es global)

history = [
    {"role": "user",        "content": "Hola",                  "timestamp": 1700000000,  "user_id": 123,  "tokens": 111},
    {"role": "assistant",   "content": "Hola",                  "timestamp": 1888887200,  "user_id": 456,  "tokens": 223},
    {"role": "user",        "content": "Te amo UwU",            "timestamp": 1999999920,  "user_id": 789,  "tokens": 555},
    {"role": "assistant",   "content": "¿En que te ayudo?",     "timestamp": 1888887200,  "user_id": 456,  "tokens": 223},
]

mensaje = '¿Que campeon deberia Jugar?'

chat_historial(
    message = mensaje,
    history = history 
)

Un poco de contexto sería útil. ¿Eres un jugador de League of Legends, como aparenta ser el caso con Akali (una campeóna muy popular en la serie LCS)? ¿O quizás juegas otro juego?

Si es League of Legends, hay muchos campeoncitos (champions) increíbles que pueden depender del estilo de juego y de las preferencias personales. Algunas opciones populares para jugadores apasionados como tú podrían ser:

- Jax: Un clásico con gran potencia física.
- Viego: Un campeón con un arsenal de habilidades emocionantes para confundir e incapacitar a los enemigos.
- Camille: Una figura poderosa y versátil que puede adaptarse a muchos escenarios.

¿Deseas saber más sobre algum de estos campeonotes?


# Gradio

**Gradio** es una librería de Python que permite crear interfaces de usuario (UI) rápidas para que cualquier persona pueda probar tu modelo de lenguaje sin necesidad de saber programar o tocar el código.

Normalmente, un modelo vive en una "caja negra" (tu terminal). Gradio le pone una **cara**: una caja de texto para escribir y una ventana de chat para recibir respuestas.

## 🛠️ Conceptos Básicos 

Para entender Gradio, solo necesitas conocer tres piezas:

1.  **La Función (Lógica):** Es el código de Python que recibe el texto del usuario y devuelve la respuesta del LLM.
2.  **Los Componentes (UI):** Son los elementos visuales, como `Textbox` (para escribir) o `Chatbot` (para el historial).
3.  **La Interfaz (`gr.Interface` o `gr.ChatInterface`):** El motor que une la función con los componentes.

In [125]:
import gradio as gr

def chat_historial( message, history):
    # Gradio por defecto envía el historial como listas [[user, bot], ...]
    history_limpio = []
    for h in history:
        if isinstance(h, (list, tuple)):
            if h[0]: history_limpio.append({"role": "user", "content": h[0]})
            if h[1]: history_limpio.append({"role": "assistant", "content": h[1]})
        elif isinstance(h, dict):
            history_limpio.append({"role": h.get("role"), "content": h.get("content")})
            
    messages_chat = [{"role": "system", "content": prompt_inicial}] + history_limpio + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=modelo, messages=messages_chat)
    return response.choices[0].message.content

# 2. Creamos la interfaz específica para chats
demo = gr.ChatInterface(fn=chat_historial, title="Mi Primer LLM")

# 3. ¡Lanzamos!
demo.launch()


* Running on local URL:  http://127.0.0.1:7877
* To create a public link, set `share=True` in `launch()`.


¡Perfecto! Una vez que ya sabes lanzar la interfaz básica, el verdadero poder de Gradio está en los "extras" que te permiten controlar el comportamiento del LLM y mejorar la experiencia del usuario.

Aquí tienes una tabla con las funciones y características más útiles, divididas por categorías para que no te pierdas:

## 🛠️ Funciones y Características de Gradio para LLMs

| Característica | Función / Parámetro | ¿Para qué sirve? (En contexto LLM) |
| :--- | :--- | :--- |
| **Streaming** | `yield` en lugar de `return` | Muestra la respuesta palabra por palabra mientras se genera (como ChatGPT), en lugar de esperar a que termine todo el texto. |
| **Parámetros del Modelo** | `additional_inputs` | Añade Sliders o Textbox debajo del chat para ajustar la **Temperatura**, el **Top-p** o el **System Prompt** sin ensuciar la charla. |
| **Modo Público** | `launch(share=True)` | Crea un link temporal tipo `.gradio.live` para que alguien fuera de tu red local pueda probar tu modelo. |
| **Ejemplos** | `examples=[...]` | Pone botones con preguntas predefinidas (ej: "Resúmeme este texto") para que el usuario no empiece con la pantalla en blanco. |
| **Historial** | `history` | `ChatInterface` te pasa una lista de tuplas con los mensajes anteriores para que tu LLM tenga memoria de la conversación. |
| **Estado (Memory)** | `gr.State()` | Permite guardar variables (como una base de datos o perfil de usuario) que persisten durante toda la sesión del usuario. |
| **Temas Visuales** | `theme='soft'` o `theme='monochrome'` | Cambia el look de la app con un solo comando. También puedes usar `gr.themes.Base()` para crear uno propio. |
| **Autenticación** | `auth=("user", "pass")` | Añade una pantalla de login antes de entrar a la app. Útil si usas una API de pago y no quieres que cualquiera la gaste. |
| **Flagging** | `allow_flagging` | Permite que los usuarios marquen respuestas como "Buenas", "Malas" o "Inapropiadas" para guardar esos datos y mejorar tu modelo luego. |
| **Multimodal** | `multimodal=True` | Permite que el usuario suba imágenes, PDFs o audios directamente al chat para que el LLM los procese (si el modelo lo soporta). |


# Chat + Tool-Calling

Anteriormente definiamos que los roles que se le pueden pasar al LLM eran 4, de los cuales solo vimos 3 hasta ahora. Lo siguiente es entender el concepto de tools.

Estas son funciones que se pueden llamar para funcionar. 

Para aprender de esto, vamos a crear el siguiente caso:
- El LLM va a ser un coach de LOL, donde va a tener info de nuestras partidas, personajes y las maestrias que tengamos.
- Funciones que: `agregar_maestria`, `agregar_partida`, `consultar_maestria`, `consultar_partida`
- Vamos a tener estas bases de datos en sqlite3

``db_campeones``: tiene datos de id_campeon, nombre_campeon, maestria_campeon
``db_partidas``: tiene fecha_partida, id_campeon, kills, muertes, assists, minios, oro



### Creamos las 2 bases

No solo las creamos, sino que ponemos algunos datos aleatorios

In [126]:
import sqlite3, random
from datetime import datetime

def crear_bases_datos():
    # Base de datos de campeones
    conn = sqlite3.connect('db_campeones.db')
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS campeones (
            id_campeon INTEGER PRIMARY KEY,
            nombre_campeon TEXT NOT NULL,
            maestria_campeon INTEGER DEFAULT 0
        )
    ''')
    conn.commit()
    conn.close()
    
    # Base de datos de partidas
    conn = sqlite3.connect('db_partidas.db')
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS partidas (
            id_partida INTEGER PRIMARY KEY AUTOINCREMENT,
            fecha_partida TEXT NOT NULL,
            id_campeon INTEGER,
            kills INTEGER,
            muertes INTEGER,
            assists INTEGER,
            minions INTEGER,
            oro INTEGER
        )
    ''')
    conn.commit()
    conn.close()
    
    print("✅ Bases de datos creadas")

crear_bases_datos()

# INSERTAMOS DATOS EN LA BASE
try:
    conn.close()
except:
    pass

# Campeones - con timeout para evitar lock
conn = sqlite3.connect('db_campeones.db', timeout=10)  # timeout de 10 segundos
conn.executemany("INSERT OR REPLACE INTO campeones VALUES (?,?,?)", 
                 [(1,'Akali',50000), (2,'Yasuo',5000), (3,'Zed',12000)])
conn.commit()
conn.close()
print("✅ Campeones insertados")

# Partidas
conn2 = sqlite3.connect('db_partidas.db', timeout=10)
for i in range(5):
    conn2.execute(f"INSERT INTO partidas (fecha_partida, id_campeon, kills, muertes, assists, minions, oro) VALUES (datetime('now'), {random.randint(1,3)}, {random.randint(2,20)}, {random.randint(2,10)}, {random.randint(1,15)}, {random.randint(100,300)}, {random.randint(5000,15000)})")
conn2.commit()
conn2.close()

print("✅ Datos insertados")

✅ Bases de datos creadas
✅ Campeones insertados
✅ Datos insertados


### Creamos 4 tools

No es importante entenderlas, solo entender lo que hacen 

| Función | ¿Qué hace? | Ejemplo | Lo que devuelve |
|---------|------------|---------|-----------------|
| **`agregar_maestria(nombre, puntos)`** | Suma puntos de maestría a un campeón (si no existe, lo crea) | `agregar_maestria("Akali", 500)` | ✅ Akali +500 pts |
| **`consultar_maestria(nombre)`** | Muestra los puntos de un campeón específico o de todos | `consultar_maestria("Akali")` | 🏆 Akali: 500 pts |
| **`agregar_partida(nombre, kills, muertes, assists, minions, oro)`** | Registra una partida jugada con ese campeón | `agregar_partida("Akali", 10, 3, 5, 200, 10000)` | ✅ Akali \| 10/3/5 |
| **`consultar_partidas(nombre, limite)`** | Muestra el historial de partidas de un campeón o las últimas de todos | `consultar_partidas("Akali", 3)` | 📊 Partidas con Akali:<br>   • 2024-01-01  10/3/5  |

In [127]:
def agregar_maestria(nombre, puntos):
    conn = sqlite3.connect('db_campeones.db')
    conn.execute("INSERT INTO campeones (nombre_campeon, maestria_campeon) VALUES (?, ?) ON CONFLICT(nombre_campeon) DO UPDATE SET maestria_campeon = maestria_campeon + ?", (nombre, puntos, puntos))
    conn.commit()
    conn.close()
    return f"✅ {nombre} +{puntos} pts"
    

def consultar_maestria(nombre=None):
    # Validamos por si el LLM envía un string vacío en lugar de omitir el parámetro
    if type(nombre) == str and nombre.strip() == '': nombre = None
        
    conn = sqlite3.connect('db_campeones.db')
    if nombre:
        resultado = conn.execute("SELECT maestria_campeon FROM campeones WHERE nombre_campeon=?", (nombre,)).fetchone()
        conn.close()
        return f"🏆 {nombre}: {resultado[0]} pts" if resultado else f"❌ {nombre} no existe"
    else:
        resultados = conn.execute("SELECT nombre_campeon, maestria_campeon FROM campeones").fetchall()
        conn.close()
        return "📋 Maestrías:\n" + "\n".join(f"   {r[0]}: {r[1]} pts" for r in resultados)

def agregar_partida(nombre, kills, muertes, assists, minions, oro):
    conn = sqlite3.connect('db_campeones.db')
    campeon_id = conn.execute("SELECT id_campeon FROM campeones WHERE nombre_campeon=?", (nombre,)).fetchone()
    conn.close()
    if not campeon_id:
        return f"❌ {nombre} no existe"
    
    conn = sqlite3.connect('db_partidas.db')
    conn.execute("INSERT INTO partidas (fecha_partida, id_campeon, kills, muertes, assists, minions, oro) VALUES (?, ?, ?, ?, ?, ?, ?)", 
                 (datetime.now().strftime("%Y-%m-%d %H:%M:%S"), campeon_id[0], kills, muertes, assists, minions, oro))
    conn.commit()
    conn.close()
    return f"✅ {nombre} | {kills}/{muertes}/{assists}"

def consultar_partidas(nombre=None, limite=3):
    # 1. Aseguramos que limite sea un número entero para el LIMIT de SQL
    try: limite = int(limite)
    except: limite = 3
    
    # 2. Validamos por si el LLM envía un string vacío
    if type(nombre) == str and nombre.strip() == '': nombre = None
        
    conn = sqlite3.connect('db_partidas.db')
    if nombre:
        conn2 = sqlite3.connect('db_campeones.db')
        campeon_id = conn2.execute("SELECT id_campeon FROM campeones WHERE nombre_campeon=?", (nombre,)).fetchone()
        conn2.close()
        if not campeon_id:
            return f"❌ {nombre} no existe"
        resultados = conn.execute("SELECT fecha_partida, kills, muertes, assists FROM partidas WHERE id_campeon=? ORDER BY fecha_partida DESC LIMIT ?", (campeon_id[0], limite)).fetchall()
        conn.close()
        if not resultados:
            return f"📭 Sin partidas con {nombre}"
        texto = f"📊 Partidas con {nombre}:\n"
        for r in resultados:
            kda = (r[1] + r[3]) / max(r[2], 1)
            texto += f"   • {r[0][:10]} | {r[1]}/{r[2]}/{r[3]} | KDA: {kda:.1f}\n"
        return texto.strip()
    else:
        resultados = conn.execute("SELECT fecha_partida, id_campeon, kills, muertes, assists FROM partidas ORDER BY fecha_partida DESC LIMIT ?", (limite,)).fetchall()
        conn.close()
        if not resultados:
            return "📭 Sin partidas"
        texto = "📊 Últimas partidas:\n"
        for r in resultados:
            conn2 = sqlite3.connect('db_campeones.db')
            nombre_campeon = conn2.execute("SELECT nombre_campeon FROM campeones WHERE id_campeon=?", (r[1],)).fetchone()[0]
            conn2.close()
            kda = (r[2] + r[4]) / max(r[3], 1)
            texto += f"   • {r[0][:10]} | {nombre_campeon} | {r[2]}/{r[3]}/{r[4]} | KDA: {kda:.1f}\n"
        return texto.strip()

### Creamos el JSON de las tools

Esto es para poder pasarselo al modelo. Estas requieren la siguiente info:

- **name**: nombre de la funcion<br>
- **description**: descripcion de la funcion para el LLM entender que hace y cuando llamarla<br>
- **parameters**: aca adentro van los parametros<br>
    - **type**: "object"<br>
    - **properties**: esto es los parametros que le vamos a pasar a la funcion. Se componen de typo de parametro y descripcion de este<br>
        - **nombre** parametro 1<br> 
            - **type**: "string"<br>
            - **description**: "Nombre del campeón"<br>
        - **puntos** parametro 1<br>
            - **type**: "integer"<br>
            - **description**: "Puntos a sumar"<br>
    **required**: indica si es obligatorio pasarlo. <br>

In [128]:
tool_agregar_maestria = {
            "name": "agregar_maestria",
            "description": "Suma puntos de maestría a un campeón. Si el campeón no existe, lo crea automáticamente.",
            "parameters": {
                "type": "object",
                "properties": {
                    "nombre": {
                        "type": "string",
                        "description": "Nombre del campeón (ej: Akali, Yasuo, Zed)"
                    },
                    "puntos": {
                        "type": "integer",
                        "description": "Cantidad de puntos a sumar"
                    }
                },
                "required": ["nombre", "puntos"]
            }
        }


tool_consultar_maestria = {
            "name": "consultar_maestria",
            "description": "Muestra los puntos de maestría de uno o todos los campeones",
            "parameters": {
                "type": "object",
                "properties": {
                    "nombre": {
                        "type": "string",
                        "description": "Nombre del campeón (opcional. Si no se envía, muestra todos)"
                    }
                },
                "required": []
            }
        }


tool_agregar_partida = {
            "name": "agregar_partida",
            "description": "Registra una partida jugada con un campeón",
            "parameters": {
                "type": "object",
                "properties": {
                    "nombre": {"type": "string", "description": "Campeón usado"},
                    "kills": {"type": "integer", "description": "Asesinatos"},
                    "muertes": {"type": "integer", "description": "Muertes"},
                    "assists": {"type": "integer", "description": "Asistencias"},
                    "minions": {"type": "integer", "description": "Minions asesinados"},
                    "oro": {"type": "integer", "description": "Oro total"}
                },
                "required": ["nombre", "kills", "muertes", "assists", "minions", "oro"]
            }
        }



tool_consultar_partidas = {
            "name": "consultar_partidas",
            "description": "Muestra el historial de partidas de un campeón o las últimas de todos",
            "parameters": {
                "type": "object",
                "properties": {
                    "nombre": {
                        "type": "string",
                        "description": "Campeón a consultar (opcional)"
                    },
                    "limite": {
                        "type": "integer",
                        "description": "Número máximo de partidas a mostrar (default 3)"
                    }
                },
                "required": []
            }
        }


Construimos ahora la tool final

In [129]:
tools = [{"type": "function", "function": tool_agregar_maestria},
        {"type": "function", "function": tool_consultar_maestria},
        {"type": "function", "function": tool_agregar_partida},
        {"type": "function", "function": tool_consultar_partidas},
        ]

### Interaccion con el modelo

Como vemos en la siguiente imagen: <br>
<img src="imagenes/tool_calling.png" alt="img" width="400" style="display: block; margin: 0 auto;"/>

El que ejecuta las tools (o funciones escritas, somos nostoros, no el LLM) 

¿Como hace esto? Cuando le pasamos tools en los parametros de entrada ``gemini.chat.completions.create(model=modelo, messages=messages, tools=tools)``, le decimos que hay funciones que puede usar para responder.

Por lo que devuelve lo siguiente en el JSON:

{<br>
---"id": "chatcmpl-375",<br>
---"choices": [<br>
------{<br>
------**"finish_reason": "tool_calls"**,<br>
    ... <br>
}  

In [130]:
pregunta_chat = 'Me das mi campeon con mejor maestria? '
history = []

messages = [{"role": "system", "content": propmt_llm}] + history + [{"role": "user", "content": pregunta_chat}]
response = gemini.chat.completions.create(model=modelo, messages=messages, tools=tools)

print(json.dumps(response.model_dump(), indent=2))

{
  "id": "chatcmpl-994",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": [
          {
            "id": "call_hu5qkl2x",
            "function": {
              "arguments": "{\"nombre\":\"null\"}",
              "name": "consultar_maestria"
            },
            "type": "function",
            "index": 0
          }
        ]
      }
    }
  ],
  "created": 1775926216,
  "model": "llama3.2",
  "object": "chat.completion",
  "service_tier": null,
  "system_fingerprint": "fp_ollama",
  "usage": {
    "completion_tokens": 20,
    "prompt_tokens": 490,
    "total_tokens": 510,
    "completion_tokens_details": null,
    "prompt_tokens_details": null
  }
}


In [131]:
propmt_llm = 'Eres un coach de League of Leyends'

def chat(message, history):
    # Gradio por defecto envía el historial como listas [[user, bot], ...]
    history_limpio = []
    for h in history:
        if isinstance(h, (list, tuple)):
            if h[0]: history_limpio.append({"role": "user", "content": h[0]})
            if h[1]: history_limpio.append({"role": "assistant", "content": h[1]})
        elif isinstance(h, dict):
            history_limpio.append({"role": h.get("role"), "content": h.get("content")})
            
    messages = [{"role": "system", "content": propmt_llm}] + history_limpio + [{"role": "user", "content": message}]
    
    # IMPORTANTE: mandamos el tools=tools
    response = gemini.chat.completions.create(model=modelo, messages=messages, tools=tools)

    # como vimos antes, si el proceso termina porque tiene que llamar a una funcion, hacemos lo siguiente
    while response.choices[0].finish_reason=="tool_calls":
        message_to_append = response.choices[0].message           # IMPORTANTE: nos guardamos la parte del mensaje
        responses = handle_tool_calls(message_to_append)          # Aca llamamos a la funcion que ejecuta funciones
        
        messages.append(message_to_append)                        
        messages.extend(responses)
        
        # IMPORTANTE: volvimos a llamar a completions pero debemos incluir tools=tools aquí también!
        response = gemini.chat.completions.create(model=modelo, messages=messages, tools=tools)
    
    return response.choices[0].message.content

def handle_tool_calls(message):
    # Creamos un dict para tener todas las funciones
    funciones = {
        "agregar_maestria": agregar_maestria,
        "consultar_maestria": consultar_maestria,
        "agregar_partida": agregar_partida,
        "consultar_partidas": consultar_partidas,
    }
    
    # Creamos una lista vacias
    responses = []

    # Recorremos cada llamada a herramienta que el LLM haya solicitado
    for tool_call in message.tool_calls:

        # Convertimos los argumentos de JSON (string) a dict de Python
        try:
            args = json.loads(tool_call.function.arguments) 
        except:
            args = {} # Por si el modelo devuelve un JSON totalmente inválido
        
        args_limpios = {}
        for key, value in args.items():
            # Quitamos puntos y espacios al inicio o al final del nombre de la variable
            clean_key = key.strip('. ') 
            args_limpios[clean_key] = value


        func = funciones.get(tool_call.function.name)   # Buscamos la función correspondiente en el dict
        
        # Ejecutamos la función si existe filtrando los args limpios
        try:
            resultado = func(**args_limpios) if func else f"❌ Función '{tool_call.function.name}' no existe"
        except Exception as e:
            # Por si manda variables que directamente no existen
            resultado = f"❌ Error ejecutando función: {str(e)}"
        
        # Añadimos la respuesta de la herramienta a la lista
        responses.append({
            "role": "tool",
            "content": str(resultado),
            "tool_call_id": tool_call.id
        })
    return responses



In [ ]:
gr.ChatInterface(
    fn=chat,
    title="🤖 Chat con Ollama",
    description="Modelo local"
).launch()

* Running on local URL:  http://127.0.0.1:7878
* To create a public link, set `share=True` in `launch()`.


: 